In [ ]:
import os
import re
import pandas as pd
import plotly.express as px

from typing import Union

import pandas as pd
import plotly.express as px

import plotly.io as pio
pio.renderers.default = "notebook"

In [ ]:
def plot_metrics(
  d: pd.DataFrame, 
  id_vars: list = ["bids_name", "site", "sub", "ses"],
  imaging_log: Union[str, bytes, os.PathLike] = os.path.join('/home', 'psadil', 'Documents', 'git', 'a2cps', 'mri_imaging_pipeline', 'aggregator_qc_app', 'tests', 'imaging_log.csv'),
  height: float = 1080*2,
  scanners: list = ['UI', 'NS', 'UC', 'UM']) -> str:  
  
  d.drop(d.filter(regex='spacing.*|size.*').columns, axis=1, inplace=True)
  
  dind = d[['bids_name']].copy()
  dind['sub'] = [int(re.findall('sub-(\d+)', x)[0]) for x in dind['bids_name']]
  dind['ses'] = [re.findall('ses-([V|v]\d)', x)[0] for x in dind['bids_name']]
  hover_data = ["ses"]
  if "run" in id_vars:
    hover_data += ["run"]
    dind['run'] = [re.findall('run-(\d+)', x)[0] for x in dind['bids_name']]

  sites = (
    pd.read_csv(imaging_log, usecols=['subject_id', 'site'])
    .rename(columns={'subject_id':'sub'})
    .drop_duplicates()
    .query("site in @scanners"))
  
  dind = dind.merge(sites, on='sub', how='left')
  d2 = pd.melt(d.merge(dind, on=["bids_name"]), id_vars=id_vars, var_name="metric").dropna()
  d2['sub'] = d2['sub'].apply(lambda x: int(str(x)[1:])).copy()

  # d2 = d2[d2['metric'].str.contains('summary')]

  fig = px.scatter(
    d2.sort_values(by=['site','sub']), 
    x="sub", 
    y="value", 
    symbol="site", 
    color="site", 
    facet_col="metric", 
    hover_data=hover_data, 
    # marginal_x="histogram", 
    facet_col_wrap=6,
    facet_row_spacing=0.02,
    facet_col_spacing=0.04)
  fig.update_yaxes(matches=None, showticklabels=True)
  fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
  fig.update_layout(autosize=False, height=height, width=1500)

  return fig

# T1w

In [ ]:
anat_outliers = plot_metrics(d=pd.read_csv("/home/psadil/Documents/git/a2cps/mri_imaging_pipeline/aggregator_qc_app/tests/group_T1w.tsv", delimiter="\t"), scanners=['NS','UI','UM'])
anat_outliers.show()

## T1w - UC

In [ ]:
anat_outliers = plot_metrics(d=pd.read_csv("/home/psadil/Documents/git/a2cps/mri_imaging_pipeline/aggregator_qc_app/tests/group_T1w.tsv", delimiter="\t"), scanners=['UC'])
anat_outliers.show()

# bold

In [ ]:
bold = pd.read_csv("/home/psadil/Documents/git/a2cps/mri_imaging_pipeline/aggregator_qc_app/tests/group_bold.tsv", delimiter="\t")
cuff = bold[bold['bids_name'].str.contains('cuff')].copy()
rest = bold[bold['bids_name'].str.contains('rest')].copy()

## cuff

In [ ]:
plot_metrics(cuff, id_vars=["bids_name", "site", "sub", "ses", "run"], scanners=['NS','UI','UM']).show()

### cuff - UC

In [ ]:
plot_metrics(cuff, id_vars=["bids_name", "site", "sub", "ses", "run"], scanners=['UC']).show()

## rest

In [ ]:
plot_metrics(rest, id_vars=["bids_name", "site", "sub", "ses", "run"], scanners=['NS','UI','UM']).show()

### rest - UC

In [ ]:
plot_metrics(rest, id_vars=["bids_name", "site", "sub", "ses", "run"], scanners=['UC']).show()

# dwi

In [ ]:
dwi = (
  pd.read_csv("/home/psadil/Documents/git/a2cps/mri_imaging_pipeline/aggregator_qc_app/tests/group_dwi.csv")
  .rename(columns={"file_name": "bids_name"})
  .drop(columns=["subject_id", "acq_id", "task_id", "dir_id", "space_id", "rec_id", "session_id", "run_id"])
  )
plot_metrics(dwi).show()